In [ ]:
!nvidia-smi

Fri Dec 12 06:50:46 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ============================================================
# ✅ 0. Install & Import (Fast + Stable)
# ============================================================

!pip install --upgrade transformers datasets accelerate

import os
os.environ["WANDB_DISABLED"] = "true"   # Disable wandb forever

import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    RobertaTokenizerFast,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, precision_recall_fscore_support


# ============================================================
# ✅ 1. Load Dataset (TweetEval Hate)
# ============================================================

dataset = load_dataset("tweet_eval", "hate")


# ============================================================
# ✅ 2. Tokenizer (DistilRoBERTa)
# ============================================================

tokenizer = RobertaTokenizerFast.from_pretrained("distilroberta-base")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(tokenize, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["text"])
tokenized_dataset.set_format("torch")


# ============================================================
# ✅ 3. Metrics
# ============================================================

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="weighted", zero_division=0
    )
    acc = accuracy_score(labels, preds)

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }


# ============================================================
# ✅ 4. Load Model (FAST + GPU SAFE)
# ============================================================

model = AutoModelForSequenceClassification.from_pretrained(
    "distilroberta-base",
    num_labels=2
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)   # ✅ Force model to GPU


# ============================================================
# ✅ 5. TrainingArguments (Auto‑Fix for old versions)
# ============================================================

import inspect
args = inspect.signature(TrainingArguments).parameters

if "evaluation_strategy" in args:
    eval_arg = {"evaluation_strategy": "epoch"}
else:
    eval_arg = {"eval_strategy": "epoch"}

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=1,               # ✅ Fast
    per_device_train_batch_size=16,   # ✅ Optimized for T4
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="no",
    **eval_arg
)


# ============================================================
# ✅ 6. Trainer
# ============================================================

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics
)


# ============================================================
# ✅ 7. Train (3–5 minutes on T4)
# ============================================================

trainer.train()


# ============================================================
# ✅ 8. Evaluate
# ============================================================

results = trainer.evaluate()
print("\n✅ Final Evaluation:", results)


# ============================================================
# ✅ 9. Predict on Custom Text (GPU SAFE)
# ============================================================

def predict(text):
    model.eval()
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}  # ✅ Move to GPU

    with torch.no_grad():
        outputs = model(**inputs)

    pred = outputs.logits.argmax(dim=1).item()
    return "Hate Speech" if pred == 1 else "Not Hate Speech"


print("\n🔍 Sample Prediction:")
print(predict("I hate you"))
print(predict("Have a nice day"))


Map:   0%|          | 0/9000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2970 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at distilroberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.384800,0.501922,0.748000,0.762170,0.748000,0.749291



✅ Final Evaluation: {'eval_loss': 0.5019224286079407, 'eval_accuracy': 0.748, 'eval_precision': 0.7621701931738566, 'eval_recall': 0.748, 'eval_f1': 0.7492905494082093, 'eval_runtime': 3.4961, 'eval_samples_per_second': 286.03, 'eval_steps_per_second': 18.02, 'epoch': 1.0}

🔍 Sample Prediction:
Not Hate Speech
Not Hate Speech
